In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import numpy as np
import json
import pandas as pd
import gradio as gr
import os
import cv2
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
def normalize_image(img):
    img = img / 255.0
    return img
def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    
    img = cv2.GaussianBlur(img, (5,5), 0)
    
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    l = clahe.apply(l)
    img = cv2.merge((l,a,b))
    img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)
    
    img = normalize_image(img)

    return img

In [ ]:
datagen = ImageDataGenerator(
    preprocessing_function=lambda img: preprocess_image(img),
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)



In [ ]:
train = datagen.flow_from_directory("/kaggle/input/datasets/leshray0211/dataset65/Comprehensive Disaster Dataset(CDD)/CDD_Augmented", target_size=(128,128),
                                      batch_size=32, class_mode='categorical', subset='training')

val = datagen.flow_from_directory("/kaggle/input/datasets/leshray0211/dataset65/Comprehensive Disaster Dataset(CDD)/CDD_Augmented", target_size=(128,128),
                                    batch_size=32, class_mode='categorical', subset='validation')

In [ ]:
print(train.class_indices)

In [ ]:
data_augmentation = tf.keras.Sequential(
  [
    # Input(shape=(128,128,3)),
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
  ]
)

In [ ]:
model = Sequential([
    Input(shape=(128,128,3)),
    data_augmentation,
    Conv2D(32,(3,3),activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),
    layers.Dropout(0.2),
    Flatten(),
    Dense(128,activation='relu'),
    Dense(train.num_classes,activation='softmax')
])

In [ ]:
# model = Sequential([
#   data_augmentation,
#   Conv2D(32,(3,3),activation='relu'),
#   layers.MaxPooling2D(2,2),
#   Conv2D(32,(3,3),activation='relu'),
#   layers.MaxPooling2D(2,2),
#   Conv2D(32,(3,3),activation='relu'),
#   layers.MaxPooling2D(2,2),
#   layers.Dropout(0.2),
#   layers.Flatten(),
#   layers.Dense(128, activation='relu'),
#   layers.Dense(train.num_classes,activation='softmax')
# ])



In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
model.fit(train, validation_data=val, epochs=3)

In [ ]:
loss, acc = model.evaluate(val)

In [ ]:
img = load_img("/kaggle/input/datasets/leshray0211/dataset65/Comprehensive Disaster Dataset(CDD)/CDD_Augmented/Non_Damage/10004.jpg", target_size=(128,128))
img = img_to_array(img)/255.0
img = np.expand_dims(img, axis=0)

In [ ]:
pred = model.predict(img)
pred_class = int(np.argmax(pred))
confidence = float(np.max(pred))

In [ ]:
pred

In [ ]:
def className(n):
    if n == 0:
        return "Damaged_Infrastrucutre"
    elif n == 1:
        return "Fire_Disaster"
    elif n == 2:
        return "Human_Damage"
    elif n == 3:
        return "Land_Disaster"
    elif n == 4:
        return "Non_Damage"
    else :
        return "Water_Disaster"

In [ ]:
def predict(img):
    img = load_img(img, target_size=(128,128))
    img = img_to_array(img)/255.0
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img)
    pred_class = int(np.argmax(pred))
    return className(pred_class)

In [ ]:
demo = gr.Interface(
    fn=predict, 
    inputs=gr.Image(type="filepath"), 
    outputs=gr.Label()
)
demo.launch()